<a href="https://colab.research.google.com/github/Eman-Adly/Eman-Adly/blob/main/API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

In [3]:
df = pd.read_csv('/content/updated_book_recommendation_data.csv')

In [4]:
df

,student_id,book_id,author,title,rating,department,genre,year_of_study,published_year,num_pages,reading_frequency
0,8538,1544,Kyle Alvarado,Maintain interest manage evidence,2,Mathematics,Fantasy,4,2000,165,High
1,6131,2685,Chloe Medina,Reality free,2,Medicine,Science Fiction,2,1963,589,Low
2,1734,7678,William Edwards,Prepare management kitchen cut,5,Engineering,Educational,1,2007,751,High
3,2108,9053,Todd Steele,Modern moment provide receive image,2,Business,Mystery,4,1989,399,Low
4,3772,1641,Lisa Joseph,Traditional establish couple usually,5,History,Science Fiction,3,2018,528,Medium
...,...,...,...,...,...,...,...,...,...,...,...
1995,3844,8390,James Cherry,Hospital then,2,Physics,Non-Fiction,2,1967,912,Medium
1996,9831,3759,Anthony Haley,Laugh stuff there home sense,5,History,Thriller,1,1997,789,Medium
1997,1259,1047,Maria Adams,Step let entire reality course,3,Computer Science,Fantasy,2,1970,614,Low
1998,8720,8976,Susan Shepherd,Local become,2,Philosophy,Historical,2,1974,159,Low


In [5]:
df.isnull().sum()

,0
student_id,0
book_id,0
author,0
title,0
rating,0
department,0
genre,0
year_of_study,0
published_year,0
num_pages,0


In [6]:
df.duplicated().sum()

0

In [7]:
df.dropna(inplace=True)

In [8]:
interaction_matrix = df.pivot(index="student_id", columns="title", values="rating").fillna(0)

In [9]:

interaction_sparse = csr_matrix(interaction_matrix)
student_similarity = cosine_similarity(interaction_sparse)

student_sim_df = pd.DataFrame(student_similarity, index=interaction_matrix.index, columns=interaction_matrix.index)


def recommend_books_collaborative(student_id, student_department, student_year, top_n=5):

    if student_id not in interaction_matrix.index:
        print(" New Student ")
        top_books = df[df["department"] == student_department].sort_values(by="rating", ascending=False).head(top_n)
        return top_books[["author", "title", "genre", "published_year", "num_pages", "rating"]]


    similar_students = student_sim_df[student_id].sort_values(ascending=False)[1:6]

    similar_students_books = interaction_matrix.loc[similar_students.index]

    book_recommendations = similar_students_books.mean(axis=0)


    books_read_by_student = interaction_matrix.loc[student_id]
    unread_books = book_recommendations[books_read_by_student == 0]

    top_recommended_books = unread_books.sort_values(ascending=False).head(top_n)

    final_recommendations = df[df["title"].isin(top_recommended_books.index)][["author", "title", "genre", "published_year", "num_pages", "rating"]].drop_duplicates()

    return final_recommendations




In [10]:
# enter the (student_id	, department , year_of_study )    old student

print(recommend_books_collaborative(8538, "Mathematics", 4))

                  author                        title            genre  \
578       Kimberly Brown                Laugh surface          Fantasy   
733        Michelle Ross  Two available continue edge  Science Fiction   
1106       Matthew Jones  Training executive sell yes        Biography   
1344         Lisa Rogers    Her message international        Self-Help   
1647  Jeffrey Mclaughlin    Agree attack suffer thank          Fantasy   

      published_year  num_pages  rating  
578             1988        685       5  
733             1980        159       4  
1106            1995        119       3  
1344            1979        476       3  
1647            1992        712       5  


In [11]:
# enter the (student_id	, department , year_of_study )    new student
print(recommend_books_collaborative(1, "Mathematics", 4))

 New Student 
                    author                                           title  \
1607    Christina Martinez  Everyone include collection remember direction   
1448      Nicholas Wilkins                     Individual father pull step   
1385  Mr. Taylor Wilkinson                     Record hear second shoulder   
1402        Kevin Robinson                         Crime worker poor cause   
1417         Lori Harrison                                 Something light   

                genre  published_year  num_pages  rating  
1607         Thriller            1966        974       5  
1448      Educational            2007        604       5  
1385      Non-Fiction            1972        403       5  
1402  Science Fiction            1997        548       5  
1417        Biography            2023        527       5  


# **Collaborative Filtering**

In [12]:
#  New Student
print(recommend_books_collaborative(3, "Engineering", 1))

 New Student 
               author                         title        genre  \
40        Stacey Shah                  World really   Historical   
924  Gabrielle Foster             Time compare meet      Mystery   
778   Kevin Alexander        Study fire movie worry  Non-Fiction   
795  Melissa Villegas           More another almost   Historical   
801       Linda Scott  Community lead become person    Biography   

     published_year  num_pages  rating  
40             1975        712       5  
924            2024        765       5  
778            2009        415       5  
795            1975        669       5  
801            2016        498       5  


In [13]:
# old student
print(recommend_books_collaborative(1734, "Engineering", 1))

                  author                        title            genre  \
578       Kimberly Brown                Laugh surface          Fantasy   
733        Michelle Ross  Two available continue edge  Science Fiction   
1106       Matthew Jones  Training executive sell yes        Biography   
1344         Lisa Rogers    Her message international        Self-Help   
1647  Jeffrey Mclaughlin    Agree attack suffer thank          Fantasy   

      published_year  num_pages  rating  
578             1988        685       5  
733             1980        159       4  
1106            1995        119       3  
1344            1979        476       3  
1647            1992        712       5  


## **API**

In [14]:
!pip install fastapi uvicorn pyngrok


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.6 MB/s eta 0:00:00


In [15]:
!ngrok config add-authtoken 2uHIwaAOnVw5L7VhJrrCUdrUunc_3HuiMBikP5shbd2z1Ljfi

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [16]:
!pip install fastapi uvicorn pyngrok nest_asyncio

In [17]:
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
import uvicorn
from pyngrok import ngrok
import nest_asyncio

app = FastAPI()

@app.get("/")
def home():
    return {"message": "API is running on Colab!"}

@app.get("/recommend")
def recommend_books():
    return {"books": ["Book 1", "Book 2", "Book 3"]}

nest_asyncio.apply()

port = 8000
public_url = ngrok.connect(port).public_url
print(f"🚀 Public URL: {public_url}")

uvicorn.run(app, host="0.0.0.0", port=port)


🚀 Public URL: https://95a8-34-28-11-104.ngrok-free.app


INFO:     Started server process [6244]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     41.129.101.126:0 - "GET / HTTP/1.1" 200 OK
INFO:     41.129.101.126:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     41.129.101.126:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     41.129.101.126:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     45.244.27.134:0 - "GET /recommend HTTP/1.1" 200 OK
INFO:     41.129.101.126:0 - "GET /recommend HTTP/1.1" 200 OK
INFO:     41.129.101.126:0 - "GET /recommend HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [6244]
